Generate Multiple Transformations: Apply a variety of transformations to your dataset (e.g., log, scaling, clipping, smoothing, etc.) and ensure each transformation can be tracked and reversed.

Reverse Transformations: After applying the transformations, build your model using the transformed data. After making predictions, reverse the transformations to bring the data back to the original scale.

Compare Predictions on Original Scale: Compare the predictions in the original scale with the actual values. Compute MSE and RMSE on the original data (not the transformed data).

Rank Transformations: Based on the calculated MSE or RMSE, rank the different transformations to evaluate their effectiveness.

In [1]:
# imports
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from funcs.process_data_funcs import (
    impute_missing_values_spline, deflate_nominal_values, apply_log_transformations,
    apply_best_transformations, cap_outliers
)
from funcs.dvc_funcs import dagshub_initialization
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import RobustScaler
# import quantile and power transformer
from sklearn.preprocessing import QuantileTransformer, PowerTransformer

In [40]:
# functions
# Import the log_transformed_df.csv file from the data folder
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import mutual_info_regression
from statsmodels.tsa.stattools import adfuller
import pandas as pd
from statsmodels.tsa.stattools import adfuller
# from funcs.machine_learning import check_stationarity, plot_series_stationarity
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd
from local_settings import settings
from fredapi import Fred
import requests


def apply_sliding_window_log(data, window_size=12):
    for col in data.columns:
        for i in range(0, len(data) - window_size + 1):
            window_data = data[col].iloc[i:i + window_size]
            logged_window = np.log1p(window_data)
            data[col].iloc[i:i + window_size] = logged_window
    return data

def apply_selective_logging(data, threshold):
    for col in data.columns:
        data[col] = np.where(data[col] > threshold, np.log1p(data[col]), data[col])
    return data

def apply_relative_transform(data):
    return data.diff().divide(data.shift(1) + 1e-9).dropna()

def apply_local_smoothing(data, window_size=5):
    return data.rolling(window=window_size).mean().dropna()

def apply_soft_clipping(data, threshold, n=1):
    for col in data.columns:
        data[col] = data[col] / (1 + (data[col] / threshold)**n)
    return data

# Use the settings dictionary
api_key = settings['api_key']
series_ids = settings['series_ids']
start_date = settings['start_date']
end_date = settings['end_date']
# Base URL for API requests
base_url = 'https://api.stlouisfed.org/fred/series/observations'
# Initialize the FRED API with your API key
fred = Fred(api_key=settings['api_key'])
# from funcs.loading_csv_functions import merge_new_data, merge_new_data_and_apply_pct_change, prepare_cpi_data, preprocess_and_merge
# from funcs.loading_csv_functions import load_and_process_cpi_data
def name(self) -> any:
    return self.attribute
#############FUNCTIONS: ###############################################################################3
# Function to fetch and prepare data
def fetch_data(series_id,frequency):
    try:
        print(f"Fetching data for {series_id}")
        data = fred.get_series(series_id, observation_start=settings['start_date'], observation_end=settings['end_date'],frequency=frequency)
        data.index = pd.to_datetime(data.index)  # Convert index to datetime
        return pd.DataFrame(data, columns=[series_id])
    except Exception as e:
        print(f"Error fetching data for {series_id}: {str(e)}")
        return pd.DataFrame()
def deflate_nominal_values(df, cpi_col_name, columns_to_deflate):
    """
    Deflates the nominal values in the specified columns of the dataframe using the CPI column.

    :param df: DataFrame containing the columns to deflate and the CPI column
    :param cpi_col_name: Name of the CPI column
    :param columns_to_deflate: List of column names to deflate
    :return: DataFrame with deflated values in the specified columns
    """
    for col in columns_to_deflate:
        df.loc[:, col] = df[col] / df[cpi_col_name] * 100
    return df
# Function to apply logarithmic transformation
def apply_log_transformations(df, columns_to_transform):
    for col in columns_to_transform:
        df[col] = 100 * np.log1p(df[col])
    return df
def cap_outliers(df, cap_factor=3.0):
    for column in df.columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - cap_factor * IQR
        upper_bound = Q3 + cap_factor * IQR
        df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
        df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    return df
def check_stationarity(data):
    """
    Perform Augmented Dickey-Fuller test to check for stationarity.
    
    Arguments:
    Pandas Series: a series of data to be checked for stationarity.
    
    Returns:
    Prints test statistics and critical values.
    """
    # Perform Augmented Dickey-Fuller test
    # Perform the test using the AIC criterion for choosing the number of lags
    print('Results of Augmented Dickey-Fuller Test:')
    adf_test = adfuller(data, autolag='AIC')  

    # Extract and print the test statistics and critical values
    adf_output = pd.Series(adf_test[0:4], 
                           index=['Test Statistic', 'p-value', '#Lags Used', 'Number of Observations Used'])
    
    for key, value in adf_test[4].items():
        adf_output['Critical Value (%s)' % key] = value
    print(adf_output)
    return adf_output


import matplotlib.pyplot as plt

def plot_series_stationarity(series, window=12):
    """
    Plot the time series, its rolling mean, and its rolling standard deviation.
    
    Arguments:
    series: Pandas Series - the time series to plot.
    window: int - the window size for calculating rolling statistics.
    """
    # Calculate rolling statistics
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()

    # Plot the statistics
    plt.figure(figsize=(14, 6))
    plt.plot(series, label='Original Series')
    plt.plot(rolling_mean, label='Rolling Mean')
    plt.plot(rolling_std, label='Rolling Std Dev')
    plt.title('Time Series Stationarity Check')
    plt.legend()
    plt.show()
import itertools
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import hvplot.pandas  # Import HvPlot for Pandas
import matplotlib.pyplot as plt
import holoviews as hv
from holoviews import dim, opts
from bokeh.plotting import show  # Import show function from Bokeh
from statsmodels.tsa.arima.model import ARIMA
import pickle

def clean_data(df):
    """
    Cleans the input DataFrame by:
    - Replacing infinities with NaN
    - Filling NaN values using backfill and forward fill
    - Interpolating any remaining NaN values
    - Ensuring all columns are numeric
    """
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)
    df.interpolate(method='linear', inplace=True)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    if df.isnull().values.any():
        df.fillna(df.mean(), inplace=True)
    return df
def check_stationarity(data):
    """
    Perform Augmented Dickey-Fuller test to check for stationarity.
    
    Arguments:
    Pandas Series: a series of data to be checked for stationarity.
    
    Returns:
    Prints test statistics and critical values.
    """
    # Perform Augmented Dickey-Fuller test
    # Perform the test using the AIC criterion for choosing the number of lags
    print('Results of Augmented Dickey-Fuller Test:')
    adf_test = adfuller(data, autolag='AIC')  

    # Extract and print the test statistics and critical values
    adf_output = pd.Series(adf_test[0:4], 
                           index=['Test Statistic', 'p-value', '#Lags Used', 'Number of Observations Used'])
    
    for key, value in adf_test[4].items():
        adf_output['Critical Value (%s)' % key] = value
    print(adf_output)
    return adf_output
def plot_series_stationarity(series, window=12):
    """
    Plot the time series, its rolling mean, and its rolling standard deviation.
    
    Arguments:
    series: Pandas Series - the time series to plot.
    window: int - the window size for calculating rolling statistics.
    """
    # Calculate rolling statistics
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()

    # Plot the statistics
    plt.figure(figsize=(14, 6))
    plt.plot(series, label='Original Series')
    plt.plot(rolling_mean, label='Rolling Mean')
    plt.plot(rolling_std, label='Rolling Std Dev')
    plt.title('Time Series Stationarity Check')
    plt.legend()
    plt.show()

from scipy.interpolate import CubicSpline
def impute_missing_values_spline(df, column):
    # Ensure the index is in datetime format and sort the data
    df = df.sort_index()
    # Extract the non-missing values to fit the spline
    known_data = df.dropna(subset=[column])
    known_index = known_data.index.map(pd.Timestamp.toordinal)  # Convert dates to ordinal
    # Fit a cubic spline using known data points
    cs = CubicSpline(known_index, known_data[column])
    # Apply the cubic spline to predict missing values
    missing_index = df[df[column].isnull()].index.map(pd.Timestamp.toordinal)
    predicted_values = cs(missing_index)
    # Fill in the missing values in the original DataFrame
    df.loc[df[column].isnull(), column] = predicted_values
    return df


def evaluate_transformations(series):
    methods = {
        'None': series,
        'Simple Differencing': series.diff().dropna(),
        'Rolling Mean Subtraction': (series - series.rolling(window=7).mean()).dropna(),
        'Rolling Mean Subtraction + Differencing': (series - series.rolling(window=7).mean()).diff().dropna()
    }           

    results = {}
    for method, transformed_series in methods.items():
        adf_result = adfuller(transformed_series)
        results[method] = (adf_result[0], adf_result[1])  # Storing the ADF statistic and p-value

    best_method = min(results, key=lambda x: results[x][0])  # Find the method with the smallest ADF statistic
    return best_method, results[best_method]

def apply_best_transformations(df):
    transformed_df = pd.DataFrame(index=df.index)
    transformation_results = {}
    for column in df.columns:
        series_data = df[column].dropna()  # Ensure no NaN values which might cause issues in computations
        best_method, (best_statistic, _) = evaluate_transformations(series_data)
        transformation_results[column] = {'Best Method': best_method, 'ADF Statistic': best_statistic}

        # Print statement to declare the column and the best transformation
        # print(f"Column: {column}, Best Method: {best_method}, ADF Statistic: {best_statistic}")
        
        if best_method == 'Simple Differencing':
            transformed_df[column] = df[column].diff().bfill()
        elif best_method == 'Rolling Mean Subtraction':
            rolling_mean = df[column].rolling(window=7).mean()
            transformed_df[column] = (df[column] - rolling_mean).bfill()
        elif best_method == 'Rolling Mean Subtraction + Differencing':
            rolling_mean = df[column].rolling(window=7).mean()
            transformed_df[column] = (df[column] - rolling_mean).diff().bfill()
        else:
            transformed_df[column] = df[column]
    
    transformation_results_df = pd.DataFrame(transformation_results).T
    transformation_results_df.to_csv('best_transformations.csv')
    return transformed_df

#########################################################################################################

def apply_sliding_window_log(data, window_size=12):
    for col in data.columns:
        # Apply sliding window logging while avoiding inplace operations that may cause unintended issues
        for i in range(window_size - 1, len(data)):
            window_data = data[col].iloc[i - window_size + 1: i + 1]
            logged_window = np.log1p(window_data)
            # Update only this window
            data[col].iloc[i - window_size + 1: i + 1] = logged_window
        # Forward and backward fill for any NaNs introduced
        data[col].fillna(method='ffill', inplace=True)
        data[col].fillna(method='bfill', inplace=True)
    return data

def apply_log_transformations(df, columns_to_transform):
    for col in columns_to_transform:
        # Protect against non-positive values by applying log1p to only positive values
        df[col] = np.where(df[col] > 0, 100 * np.log1p(df[col]), df[col])
        # Handle NaNs by forward-filling and backward-filling
        df[col].fillna(method='ffill', inplace=True)
        df[col].fillna(method='bfill', inplace=True)
    return df

def apply_selective_logging(data, threshold):
    for col in data.columns:
        # Apply log1p only to values above the threshold
        data[col] = np.where(data[col] > threshold, np.log1p(data[col]), data[col])
        # Fill NaNs to handle any missing data
        data[col].fillna(method='ffill', inplace=True)
        data[col].fillna(method='bfill', inplace=True)
    return data

def apply_local_smoothing(data, window_size=5):
    return data.rolling(window=window_size).mean().dropna()





In [ ]:
import os
import pandas as pd
import numpy as np
from best_params import xgboost_params, lightgbm_params 
from sklearn.preprocessing import StandardScaler
# from funcs.process_data_funcs import (
#     impute_missing_values_spline, deflate_nominal_values, apply_log_transformations, cap_outliers, apply_best_transformations, 
# )
from funcs.dvc_funcs import dagshub_initialization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
# import quantile and power transformer
from sklearn.preprocessing import QuantileTransformer, PowerTransformer
def save_transformed_data(data, transformations_applied, iteration):
    # Create the output directory if it doesn't exist
    output_dir = 'data/processed/testing'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Create the file name based on the transformations applied
    file_name = f"transformed_data__"+"__".join(transformations_applied)+".csv"
    # print("DEBUG 1 ")
    # print("file_name: ", file_name)
    file_path = os.path.join(output_dir, file_name)

    # Save the transformed data
    data.to_csv(file_path, index=True)
    print(f"Saved transformed data: {file_path}")



# def process_data():
# Load combined data from raw CSV
combined_data = pd.read_csv('data/raw/raw_data.csv', parse_dates=True, index_col='Date')

# Separate 'USREC' before processing
# usrec_data = combined_data['USREC']
# combined_data = combined_data.drop(columns=['USREC'])  # Exclude categorical column

# Define all transformation combinations to try
# transformation_combinations = [
#     # Basic combinations
#     ['impute'],  # CHECK
#     ['impute', 'log'],  # CHECK
#     ['impute', 'scale'],  # CHECK
#     ['impute', 'pct_change'],  # CHECK
#     ['impute', 'adf'],  # CHECK
#     ['impute', 'robust'],  # CHECK
#     ['impute', 'power'],  # CHECK
#     ['impute', 'quantile'],  # CHECK
    
#     # Two transformations
#     ['impute', 'log', 'scale'],
#     ['impute', 'log', 'pct_change'],
#     ['impute', 'log', 'adf'],
#     ['impute', 'log', 'robust'],
#     ['impute', 'log', 'power'],
#     ['impute', 'log', 'quantile'],
#     ['impute', 'scale', 'pct_change'],
#     ['impute', 'scale', 'adf'],
#     ['impute', 'scale', 'robust'],
#     ['impute', 'scale', 'power'],
#     ['impute', 'scale', 'quantile'],
#     ['impute', 'pct_change', 'adf'],
#     ['impute', 'pct_change', 'robust'],
#     ['impute', 'pct_change', 'power'],
#     ['impute', 'pct_change', 'quantile'],
#     ['impute', 'adf', 'robust'],
#     ['impute', 'adf', 'power'],
#     ['impute', 'adf', 'quantile'],
    
#     # Three transformations
#     ['impute', 'log', 'scale', 'pct_change'],
#     ['impute', 'log', 'scale', 'adf'],
#     ['impute', 'log', 'scale', 'robust'],
#     ['impute', 'log', 'scale', 'power'],
#     ['impute', 'log', 'scale', 'quantile'],
#     ['impute', 'log', 'pct_change', 'adf'],
#     ['impute', 'log', 'pct_change', 'robust'],
#     ['impute', 'log', 'pct_change', 'power'],
#     ['impute', 'log', 'pct_change', 'quantile'],
#     ['impute', 'log', 'adf', 'robust'],
#     ['impute', 'log', 'adf', 'power'],
#     ['impute', 'log', 'adf', 'quantile'],
#     ['impute', 'log', 'robust', 'power'],
#     ['impute', 'log', 'robust', 'quantile'],
#     ['impute', 'log', 'power', 'quantile'],
#     ['impute', 'scale', 'pct_change', 'adf'],
#     ['impute', 'scale', 'pct_change', 'robust'],
#     ['impute', 'scale', 'pct_change', 'power'],
#     ['impute', 'scale', 'pct_change', 'quantile'],
#     ['impute', 'scale', 'adf', 'robust'],
#     ['impute', 'scale', 'adf', 'power'],
#     ['impute', 'scale', 'adf', 'quantile'],
#     ['impute', 'scale', 'robust', 'power'],
#     ['impute', 'scale', 'robust', 'quantile'],
#     ['impute', 'scale', 'power', 'quantile'],
#     ['impute', 'pct_change', 'adf', 'robust'],
#     ['impute', 'pct_change', 'adf', 'power'],
#     ['impute', 'pct_change', 'adf', 'quantile'],
#     ['impute', 'pct_change', 'robust', 'power'],
#     ['impute', 'pct_change', 'robust', 'quantile'],
#     ['impute', 'pct_change', 'power', 'quantile'],
#     ['impute', 'adf', 'robust', 'power'],
#     ['impute', 'adf', 'robust', 'quantile'],
#     ['impute', 'adf', 'power', 'quantile'],
#     ['impute', 'robust', 'power', 'quantile'],
    
#     # Four transformations
#     ['impute', 'log', 'scale', 'pct_change', 'adf'],
#     ['impute', 'log', 'scale', 'pct_change', 'robust'],
#     ['impute', 'log', 'scale', 'pct_change', 'power'],
#     ['impute', 'log', 'scale', 'pct_change', 'quantile'],
#     ['impute', 'log', 'scale', 'adf', 'robust'],
#     ['impute', 'log', 'scale', 'adf', 'power'],
#     ['impute', 'log', 'scale', 'adf', 'quantile'],
#     ['impute', 'log', 'scale', 'robust', 'power'],
#     ['impute', 'log', 'scale', 'robust', 'quantile'],
#     ['impute', 'log', 'scale', 'power', 'quantile'],
#     ['impute', 'log', 'pct_change', 'adf', 'robust'],
#     ['impute', 'log', 'pct_change', 'adf', 'power'],
#     ['impute', 'log', 'pct_change', 'adf', 'quantile'],
#     ['impute', 'log', 'pct_change', 'robust', 'power'],
#     ['impute', 'log', 'pct_change', 'robust', 'quantile'],
#     ['impute', 'log', 'pct_change', 'power', 'quantile'],
#     ['impute', 'log', 'adf', 'robust', 'power'],
#     ['impute', 'log', 'adf', 'robust', 'quantile'],
#     ['impute', 'log', 'adf', 'power', 'quantile'],
#     ['impute', 'log', 'robust', 'power', 'quantile'],
#     ['impute', 'scale', 'pct_change', 'adf', 'robust'],
#     ['impute', 'scale', 'pct_change', 'adf', 'power'],
#     ['impute', 'scale', 'pct_change', 'adf', 'quantile'],
#     ['impute', 'scale', 'pct_change', 'robust', 'power'],
#     ['impute', 'scale', 'pct_change', 'robust', 'quantile'],
#     ['impute', 'scale', 'pct_change', 'power', 'quantile'],
#     ['impute', 'scale', 'adf', 'robust', 'power'],
#     ['impute', 'scale', 'adf', 'robust', 'quantile'],
#     ['impute', 'scale', 'adf', 'power', 'quantile'],
#     ['impute', 'scale', 'robust', 'power', 'quantile'],
#     ['impute', 'pct_change', 'adf', 'robust', 'power'],
#     ['impute', 'pct_change', 'adf', 'robust', 'quantile'],
#     ['impute', 'pct_change', 'adf', 'power', 'quantile'],
#     ['impute', 'pct_change', 'robust', 'power', 'quantile'],
#     ['impute', 'adf', 'robust', 'power', 'quantile'],
    
#     # More advanced and specialized transformations
#     ['impute', 'log_all', 'pct_change', 'scale'],  # CHECK
#     ['impute', 'log_all', 'pct_change', 'robust'],  # CHECK
#     ['impute', 'log_all', 'pct_change', 'power'],  # CHECK
#     ['impute', 'log_all', 'pct_change', 'quantile'],  # CHECK
#     ['impute', 'log_all', 'adf', 'pct_change'],
#     ['impute', 'log_all', 'adf', 'scale'],
#     ['impute', 'log_all', 'adf', 'robust'],
#     ['impute', 'log_all', 'adf', 'power'],
#     ['impute', 'log_all', 'adf', 'quantile'],
#     ['impute', 'log_all', 'scale', 'adf'],
#     ['impute', 'log_all', 'scale', 'robust'], 
#     ['impute', 'log_all', 'scale', 'power'], 
#     ['impute', 'log_all', 'scale', 'quantile'],  
#     ['impute', 'log_all', 'robust', 'power'], 
#     ['impute', 'log_all', 'robust', 'quantile'],  
#     ['impute', 'log_all', 'power', 'quantile'],
    
#     # Transformation methods
#     ['impute', 'sliding_window_log', 'pct_change'],
#     ['impute', 'selective_log', 'pct_change'],
#     ['impute', 'relative_transform', 'pct_change'],
#     ['impute', 'local_smooth', 'pct_change'],
#     ['impute', 'soft_clipping', 'pct_change'],
#     ['impute', 'sliding_window_log'],
#     ['impute', 'selective_log'],
#     ['impute', 'relative_transform'],
#     ['impute', 'local_smooth'],
#     ['impute', 'soft_clipping'],
# ]

transformation_combinations = [
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change', 'adf', 'cap'],
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change', 'adf'],
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change'],
    ['impute', 'deflate', 'log_all', 'pct_change'],
    ['impute', 'deflate', 'pct_change'],
    ['impute', 'scale', 'pct_change', 'adf'],
    ['impute', 'scale', 'adf', 'pct_change'],
    ['impute', 'adf', 'scale', 'pct_change'],
    ['impute', 'pct_change', 'adf'],
    ['impute', 'adf', 'pct_change'],
    ['impute', 'log_all', 'pct_change'],
    ['impute'],
    ['impute', 'log_all', 'impute', 'pct_change'],
    ['impute', 'log_all', 'impute', 'pct_change', 'impute', 'scale'],
    ['impute', 'log_all', 'impute', 'pct_change', 'impute', 'scale', 'impute'],

    # Basic combinations
    ['impute', 'log_all'],
    ['impute', 'scale'],
    ['impute', 'pct_change'],
    ['impute', 'adf'],
    ['impute', 'robust'],
    ['impute', 'power'],
    ['impute', 'quantile'],
    
    # Two transformations (Mix of logging alternatives)
    ['impute', 'local_smooth', 'pct_change'],
    ['impute', 'soft_clipping', 'adf'],
    ['impute', 'sliding_window_log', 'robust'],
    ['impute', 'relative_transform', 'power'],

    # Three transformations (More balanced between logging and alternatives)
    ['impute', 'log_all', 'scale', 'pct_change'],
    ['impute', 'soft_clipping', 'scale', 'adf'],
    ['impute', 'local_smooth', 'pct_change', 'robust'],
    ['impute', 'sliding_window_log', 'scale', 'power'],
    ['impute', 'relative_transform', 'scale', 'quantile'],

    # Four transformations (Advanced options with diverse methods)
    ['impute', 'log_all', 'scale', 'pct_change', 'adf'],
    ['impute', 'soft_clipping', 'scale', 'pct_change', 'robust'],
    ['impute', 'local_smooth', 'pct_change', 'scale', 'quantile'],
    ['impute', 'sliding_window_log', 'scale', 'adf', 'power'],
    ['impute', 'relative_transform', 'scale', 'robust', 'quantile'],
    
    # Smoothing and clipping as alternatives to logging
    ['impute', 'soft_clipping', 'pct_change', 'robust'],
    ['impute', 'local_smooth', 'adf', 'scale'],
    ['impute', 'sliding_window_log', 'pct_change', 'robust'],

    # Additional Diverse Options
    ['impute', 'log_all', 'soft_clipping', 'pct_change'],  # Diverse logging and clipping
    ['impute', 'log_all', 'relative_transform', 'pct_change'],  # Diverse logging with relative transform
    ['impute', 'log_all', 'sliding_window_log', 'pct_change'],  # Combination of log and sliding window log
    ['impute', 'log_all', 'local_smooth', 'pct_change'],  # Log with local smoothing
    ['impute', 'local_smooth', 'adf', 'quantile'],  # Smoothing with quantile transform
    ['impute', 'soft_clipping', 'adf', 'quantile'],  # Clipping with quantile transformation
    ['impute', 'sliding_window_log', 'adf', 'robust'],  # Sliding window with ADF and robust scaling
    ['impute', 'sliding_window_log', 'adf', 'power'],  # Sliding window with power transformation
    ['impute', 'log_all', 'local_smooth', 'adf'],  # Diverse log with smoothing
    ['impute', 'soft_clipping', 'sliding_window_log', 'adf'], # Clipping with sliding window and ADF
    ['impute', 'local_smooth', 'robust', 'quantile'],  # Diverse smoothing with robust scaling and quantile transformation
    ['impute', 'soft_clipping', 'relative_transform', 'adf'],  # Combining clipping with relative transformation and stationarity
    ['impute', 'sliding_window_log', 'scale', 'pct_change'],  # Sliding window with scaling and percentage change
    ['impute', 'log_all', 'sliding_window_log', 'scale'],  # Log all followed by sliding window and scaling
    ['impute', 'log_all', 'soft_clipping', 'adf'],  # Log all with clipping and stationarity
    ['impute', 'relative_transform', 'robust', 'power'],  # Relative transformation with robust scaling and power transformation
    ['impute', 'local_smooth', 'adf', 'robust'],  # Smoothing with stationarity and robust scaling
    ['impute', 'soft_clipping', 'pct_change', 'power'],  # Clipping with percentage change and power transformation
    ['impute', 'log_all', 'local_smooth', 'robust'],  # Log all with smoothing and robust scaling
    ['impute', 'sliding_window_log', 'pct_change', 'adf'],  # Sliding window log with percentage change and stationarity
    ['impute', 'local_smooth', 'pct_change'],  # Basic prioritized combination
    ['impute', 'local_smooth', 'pct_change', 'adf'],  # Adding stationarity check
    ['impute', 'local_smooth', 'pct_change', 'scale'],  # Scaling after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'robust'],  # Robust scaling after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'soft_clipping'],  # Clipping after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'power'],  # Power transform after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'quantile'],  # Quantile transform after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'adf', 'scale'],  # Combining stationarity check with scaling
    ['impute', 'local_smooth', 'pct_change', 'robust', 'quantile'],  # Adding quantile transform to robust scaling
    ['impute', 'local_smooth', 'pct_change', 'adf', 'power'],  # Adding power transform to stationarity check
]



# Iterate through each combination of transformations
for i, transformations in enumerate(transformation_combinations):
    data = combined_data.copy()
    transformations_applied = []

    # 1. Impute missing values
    if 'impute' in transformations:
        for column in data.columns:
            data = impute_missing_values_spline(data, column)
        transformations_applied.append('impute')

    # 2. Deflate nominal values
    if 'deflate' in transformations:
        cpi_col_name = 'CPIAUCSL'
        columns_to_deflate = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE', 
            'FGCE', 'DSPI'
        ]
        data = deflate_nominal_values(data, cpi_col_name, columns_to_deflate)
        transformations_applied.append('deflate')

    # 3. Apply logarithmic transformations
    if 'log' in transformations:
        columns_to_transform = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 
            'GCE', 'FGCE', 'HOUST', 'DSPI', 'M1', 'M1V', 'M2', 
            'WTISPLC'
        ]
        data = apply_log_transformations(data, columns_to_transform)
        transformations_applied.append('log')

    # 4. Standardize/Normalize the Data
    if 'scale' in transformations:
        scaler = StandardScaler()
        data[data.columns] = scaler.fit_transform(data)
        transformations_applied.append('scale')

    # 5. Apply Percentage Change
    if 'pct_change' in transformations:
        data = data.pct_change().dropna()
        transformations_applied.append('pct_change')

    # 6. Apply ADF-based transformations
    if 'adf' in transformations:
        data = apply_best_transformations(data)
        transformations_applied.append('adf')

    # 7. Cap outliers
    if 'cap' in transformations:
        data = cap_outliers(data, cap_factor=3.0)
        transformations_applied.append('cap')

    # 8. Robust Scaler
    if 'robust' in transformations:
        scaler = RobustScaler()
        data[data.columns] = scaler.fit_transform(data)
        transformations_applied.append('robust')

    # 9. Quantile Transformer
    if 'quantile' in transformations:
        scaler = QuantileTransformer(n_quantiles=min(len(raw_data), 1000))
        data[data.columns] = scaler.fit_transform(data)
        transformations_applied.append('quantile')
    
    # 10. Power Transformer
    if 'power' in transformations:
        scaler = PowerTransformer()
        data[data.columns] = scaler.fit_transform(data)
        transformations_applied.append('power')
    
    # 11. Log All
    if 'log_all' in transformations:
        columns_to_transform = data.columns
        data = apply_log_transformations(data, columns_to_transform)
        transformations_applied.append('log_all')
    
    # 12. Sliding Window Log Transformation
    if 'sliding_window_log' in transformations:
        data = apply_sliding_window_log(data)
        transformations_applied.append('sliding_window_log')

    # 13. Selective Log Transformation
    if 'selective_log' in transformations:
        threshold = 1.0  # Choose your threshold here
        data = apply_selective_logging(data, threshold)
        transformations_applied.append('selective_log')

    # 14. Relative Transformations (Normalized Differences)
    if 'relative_transform' in transformations:
        data = apply_relative_transform(data)
        transformations_applied.append('relative_transform')

    # 15. Local Smoothing Before Logging
    if 'local_smooth' in transformations:
        data = apply_local_smoothing(data)
        transformations_applied.append('local_smooth')

    # 16. Soft Clipping for Outliers
    if 'soft_clipping' in transformations:
        threshold = 10  # Set an appropriate threshold
        data = apply_soft_clipping(data, threshold)
        transformations_applied.append('soft_clipping')
        
    # # Add 'USREC' back to the dataset after processing
    # data['USREC'] = usrec_data.reindex(data.index)

    # Save the transformed data
    save_transformed_data(data, transformations_applied, i)






In [54]:
transformation_combinations  = [    
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change', 'adf', 'cap'],
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change', 'adf'],
    ['impute', 'deflate', 'log_all', 'scale', 'pct_change'],
    ['impute', 'deflate', 'log_all', 'pct_change'],
    ['impute', 'deflate', 'pct_change'],
    ['impute', 'scale', 'pct_change', 'adf'],
    ['impute', 'scale', 'adf', 'pct_change'],
    ['impute', 'adf', 'scale', 'pct_change'],
    ['impute', 'pct_change', 'adf'],
    ['impute', 'adf', 'pct_change'],
    ['impute', 'log_all', 'pct_change'],
    ['impute'],
    ['impute', 'log_all', 'impute', 'pct_change'],
    ['impute', 'log_all', 'impute', 'pct_change', 'impute', 'scale'],
    ['impute', 'log_all', 'impute', 'pct_change', 'impute', 'scale', 'impute'],

    # Basic combinations
    ['impute', 'log_all'],
    ['impute', 'scale'],
    ['impute', 'pct_change'],
    ['impute', 'adf'],
    ['impute', 'robust'],
    ['impute', 'power'],
    ['impute', 'quantile'],
    
    # Two transformations (Mix of logging alternatives)
    ['impute', 'local_smooth', 'pct_change'],
    ['impute', 'soft_clipping', 'adf'],
    ['impute', 'sliding_window_log', 'robust'],
    ['impute', 'relative_transform', 'power'],

    # Three transformations (More balanced between logging and alternatives)
    ['impute', 'log_all', 'scale', 'pct_change'],
    ['impute', 'soft_clipping', 'scale', 'adf'],
    ['impute', 'local_smooth', 'pct_change', 'robust'],
    ['impute', 'sliding_window_log', 'scale', 'power'],
    ['impute', 'relative_transform', 'scale', 'quantile'],

    # Four transformations (Advanced options with diverse methods)
    ['impute', 'log_all', 'scale', 'pct_change', 'adf'],
    ['impute', 'soft_clipping', 'scale', 'pct_change', 'robust'],
    ['impute', 'local_smooth', 'pct_change', 'scale', 'quantile'],
    ['impute', 'sliding_window_log', 'scale', 'adf', 'power'],
    ['impute', 'relative_transform', 'scale', 'robust', 'quantile'],
    
    # Smoothing and clipping as alternatives to logging
    ['impute', 'soft_clipping', 'pct_change', 'robust'],
    ['impute', 'local_smooth', 'adf', 'scale'],
    ['impute', 'sliding_window_log', 'pct_change', 'robust'],

    # Additional Diverse Options
    ['impute', 'log_all', 'soft_clipping', 'pct_change'],  # Diverse logging and clipping
    ['impute', 'log_all', 'relative_transform', 'pct_change'],  # Diverse logging with relative transform
    ['impute', 'log_all', 'sliding_window_log', 'pct_change'],  # Combination of log and sliding window log
    ['impute', 'log_all', 'local_smooth', 'pct_change'],  # Log with local smoothing
    ['impute', 'local_smooth', 'adf', 'quantile'],  # Smoothing with quantile transform
    ['impute', 'soft_clipping', 'adf', 'quantile'],  # Clipping with quantile transformation
    ['impute', 'sliding_window_log', 'adf', 'robust'],  # Sliding window with ADF and robust scaling
    ['impute', 'sliding_window_log', 'adf', 'power'],  # Sliding window with power transformation
    ['impute', 'log_all', 'local_smooth', 'adf'],  # Diverse log with smoothing
    ['impute', 'soft_clipping', 'sliding_window_log', 'adf'], # Clipping with sliding window and ADF
    ['impute', 'local_smooth', 'robust', 'quantile'],  # Diverse smoothing with robust scaling and quantile transformation
    ['impute', 'soft_clipping', 'relative_transform', 'adf'],  # Combining clipping with relative transformation and stationarity
    ['impute', 'sliding_window_log', 'scale', 'pct_change'],  # Sliding window with scaling and percentage change
    ['impute', 'log_all', 'sliding_window_log', 'scale'],  # Log all followed by sliding window and scaling
    ['impute', 'log_all', 'soft_clipping', 'adf'],  # Log all with clipping and stationarity
    ['impute', 'relative_transform', 'robust', 'power'],  # Relative transformation with robust scaling and power transformation
    ['impute', 'local_smooth', 'adf', 'robust'],  # Smoothing with stationarity and robust scaling
    ['impute', 'soft_clipping', 'pct_change', 'power'],  # Clipping with percentage change and power transformation
    ['impute', 'log_all', 'local_smooth', 'robust'],  # Log all with smoothing and robust scaling
    ['impute', 'sliding_window_log', 'pct_change', 'adf'],  # Sliding window log with percentage change and stationarity
    ['impute', 'local_smooth', 'pct_change'],  # Basic prioritized combination
    ['impute', 'local_smooth', 'pct_change', 'adf'],  # Adding stationarity check
    ['impute', 'local_smooth', 'pct_change', 'scale'],  # Scaling after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'robust'],  # Robust scaling after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'soft_clipping'],  # Clipping after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'power'],  # Power transform after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'quantile'],  # Quantile transform after smoothing and pct_change
    ['impute', 'local_smooth', 'pct_change', 'adf', 'scale'],  # Combining stationarity check with scaling
    ['impute', 'local_smooth', 'pct_change', 'robust', 'quantile'],  # Adding quantile transform to robust scaling
    ['impute', 'local_smooth', 'pct_change', 'adf', 'power'],  # Adding power transform to stationarity check
]

In [55]:
# Functions to reverse transformations
# Function to fetch and prepare data
import itertools
import statsmodels.api as sm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import hvplot.pandas  # Import HvPlot for Pandas
import matplotlib.pyplot as plt
import holoviews as hv
from holoviews import dim, opts
from bokeh.plotting import show  # Import show function from Bokeh
from statsmodels.tsa.arima.model import ARIMA
import pickle

def deflate_nominal_values(df, cpi_col_name, columns_to_deflate):
    """
    Deflates the nominal values in the specified columns of the dataframe using the CPI column.

    :param df: DataFrame containing the columns to deflate and the CPI column
    :param cpi_col_name: Name of the CPI column
    :param columns_to_deflate: List of column names to deflate
    :return: DataFrame with deflated values in the specified columns
    """
    for col in columns_to_deflate:
        df.loc[:, col] = df[col] / df[cpi_col_name] * 100
    return df

def reverse_deflate_nominal_values(df,cpi_col_name,columns_to_deflate):
    """
    Inflates the deflated values in the specified columns of the dataframe using the original CPI column from raw data.

    :param df: DataFrame containing the columns to inflate and the CPI column
    :param cpi_col_name: Name of the CPI column
    :param columns_to_inflate: List of column names to inflate
    :return: DataFrame with inflated values in the specified columns
    """
    # Load the original raw data
    raw_data = pd.read_csv('data/raw/raw_data.csv', parse_dates=True, index_col='Date')

    # Ensure the CPI column is correctly aligned
    original_cpi = raw_data[cpi_col_name]

    for col in columns_to_inflate:
        df.loc[:, col] = df[col] * original_cpi / 100
    
    return df


def apply_log_transformations(df, columns_to_transform):
    for col in columns_to_transform:
        # Protect against non-positive values by applying log1p to only positive values
        df[col] = np.where(df[col] > 0, 100 * np.log1p(df[col]), df[col])
        # Handle NaNs by forward-filling and backward-filling
        df[col].fillna(method='ffill', inplace=True)
        df[col].fillna(method='bfill', inplace=True)
    return df

def reverse_log_transformations(df, columns_to_transform):
    for col in columns_to_transform:
        # Reverse the log1p transformation applied earlier
        df[col] = np.where(df[col] > 0, np.expm1(df[col] / 100), df[col])
    return df

def cap_outliers(df, cap_factor=3.0):
    for column in df.columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - cap_factor * IQR
        upper_bound = Q3 + cap_factor * IQR
        df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
        df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    return df

# def reverse_cap_outliers(df,cap_factor=3.0):
#     for column in df.columns:







def reverse_best_transformations(df):
    transformed_df = pd.Series(index=df.index)
    transformation_results = {}
    raw_data = pd.read_csv('data/raw/raw_data.csv', parse_dates=True, index_col='Date')
    for column in df.columns:
        series_data = df[column].ffill()  # Ensure no NaN values which might cause issues in computations
        best_method, (best_statistic, _) = evaluate_transformations(series_data)
        transformation_results[column] = {'Best Method': best_method, 'ADF Statistic': best_statistic}

        # Print statement to declare the column and the best transformation
        # print(f"Column: {column}, Best Method: {best_method}, ADF Statistic: {best_statistic}")
        # raw_data = raw_data[column]

        if best_method == 'Simple Differencing':
            first_value = raw_data.iloc[0]
            transformed_df[column] = df[column].cumsum() + first_value
        elif best_method == 'Rolling Mean Subtraction':
            rolling_mean = raw_data[column].rolling(window=7).mean()
            transformed_df[column] = (df[column] + rolling_mean).ffill()
        elif best_method == 'Rolling Mean Subtraction + Differencing':
            rolling_mean = raw_data[column].rolling(window=7).mean()
            first_value = raw_data.iloc[0]
            transformed_df[column] = (df[column] + rolling_mean).ffill()
            transformed_df[column] = df[column].cumsum() + first_value

        else:
            transformed_df[column] = df[column]
    
    transformation_results_df = pd.DataFrame(transformation_results).T
    transformation_results_df.to_csv('best_transformations.csv')
    return transformed_df

def reverse_sliding_window_log(data, window_size=12):
    for col in data.columns:
        # Apply sliding window logging while avoiding inplace operations that may cause unintended issues
        for i in range(window_size - 1, len(data)):
            # Before:
            # window_data = data[col].iloc[i - window_size + 1: i + 1]
            # logged_window = np.log1p(window_data)
            # After
            window_data = data[col].iloc[i - window_size + 1: i + 1]
            exp_window = expm1(window_data)
            # Update only this window
            data[col].iloc[i - window_size + 1: i + 1] = exp_window
        # Forward and backward fill for any NaNs introduced
        data[col].fillna(method='ffill', inplace=True)
        data[col].fillna(method='bfill', inplace=True)
    return data

def reverse_selective_logging(data, threshold):
    for col in data.columns:
        # Apply log1p only to values above the threshold
        data[col] = np.where(data[col] > threshold, np.expm1(data[col]), data[col])
        # Fill NaNs to handle any missing data
        data[col].fillna(method='ffill', inplace=True)
        data[col].fillna(method='bfill', inplace=True)
    return data


def reverse_relative_transform(data):
    raw_data = pd.read_csv('data/raw/raw_data.csv',index_col='Date',parse_dates=True)
    columns = raw_data.columns
    for col in columns:
        initial_value = raw_data[col].iloc[0]
        # first we reverse the differencing
        data[col] = data[col].cumsum() + initial_value
        # then we reverse the division by data.shift(1)+1e-9
        data[col] * (raw_data.shift(1)+1e-9)
    return data

import pandas as pd

def reverse_rolling_mean(data, window_size=5):
    # Initialize DataFrame for reversed data
    reversed_data = pd.DataFrame(index=smoothed_data.index, columns=smoothed_data.columns)
    smoothed_data = data
    for col in smoothed_data.columns:
        smoothed_col = smoothed_data[col]
        reversed_col = pd.Series(index=smoothed_data.index, dtype=float)
        
        for i in range(len(smoothed_data)):
            # Determine the range for the window
            start_idx = max(0, i - window_size + 1)
            end_idx = i + 1
            
            # Calculate the sum of the window based on the smoothed value
            if end_idx > start_idx:
                window_mean = smoothed_col.iloc[start_idx:end_idx].mean()
                # Approximate the original value by reversing the mean effect
                if i > 0:
                    previous_value = reversed_col.iloc[i - 1] if not pd.isna(reversed_col.iloc[i - 1]) else smoothed_col.iloc[i]
                else:
                    previous_value = smoothed_col.iloc[i]
                reversed_col.iloc[i] = smoothed_col.iloc[i] * window_size - (window_mean * (window_size - 1))
        
        reversed_data[col] = reversed_col
    
    return reversed_data

def apply_soft_clipping(data, threshold, n=1):
    for col in data.columns:
        data[col] = data[col] / (1 + (data[col] / threshold)**n)
    return data

def reverse_soft_clipping(data, threshold, n=1):
    raw_data = pd.read_csv('data/raw/raw_data.csv',index_col='Date',parse_dates=True)
    for col in data.columns:
        data[col] = data[col] * (1  + (raw_data[col] / threshold)**n)


11174.129

Date
2003-01-31    11174.129
2003-02-28          NaN
2003-03-31          NaN
2003-04-30    11312.766
2003-05-31          NaN
Name: GDP, dtype: float64

In [56]:
# Reversing the transformations
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from sklearn.neighbors import KNeighborsRegressor
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data


files = os.listdir('data/processed/testing')
xgboost_params = {'max_depth': 9, 'learning_rate': 0.016810144010995204, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5865511380196138, 'colsample_bytree': 0.9177433017782509, 'reg_alpha': 0.012513778476694921, 'reg_lambda': 8.577576991968611e-05}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832469304666387, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402007213047204, 'colsample_bytree': 0.9405245932820381, 'reg_alpha': 0.000756445935013929, 'reg_lambda': 0.0002590470172405959}

for file in files:
    # Step 1: Use predictive models to create predictions on the data we used
    data = pd.read_csv(file,index_col='Date',parse_dates=True)
    X_train,y_train,X_test,y_test = train_test_split.
    # Step 2: creating a list of transformations, based on file names:
    transformations = []
    transformation_strings = ['impute','deflate','log','scale','pct_change','adf','cap','robust','quantile','power','log_all','sliding_window_log','selective_log','soft_clipping','local_smooth','relative_transform']
    for string in transformation_strings:
        if string in file:
            transformations.append(string)
    print(f" entire list: {transformations}")
    for transformation in transformations:
        print(f" individual transformation : {transformation}")
    output_dir = 'data/processed/testing'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    # Step 2.5 prepare for reversals
    raw_data = pd.read_csv('data/raw/raw_data.csv',index_col='Date',parse_dates=True)
    file_name = f"transformed_data__"+"__".join(transformations)+".csv"
    file_path = os.path.join(output_dir, file_name)
    transformed_data = pd.read_csv(file_path, parse_dates=True, index_col='Date')
    print(f"transformed data: {transformed_data.head()}")
    # Step 3: perform multiple reversals
        # 1. Impute missing values
    if 'impute' in transformations:
        skip = 'yes'
        # Since we cannot have NaN's, we will never reverse the imputation of data

    # 2. Deflate nominal values
    if 'deflate' in transformations:
        cpi_col_name = 'CPIAUCSL'
        columns_to_deflate = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE', 
            'FGCE', 'DSPI'
        ]
        # We now reverse the deflation
        data = reverse_deflate_nominal_values(data, cpi_col_name, columns_to_deflate)
        transformations_applied.append('deflate')

    # 3. Apply logarithmic transformations
    if 'log' in transformations:
        columns_to_transform = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 
            'GCE', 'FGCE', 'HOUST', 'DSPI', 'M1', 'M1V', 'M2', 
            'WTISPLC'
        ]
        data = reverse_log_transformations(data, columns_to_transform)
        transformations_applied.append('log')

    # 4. Standardize/Normalize the Data
    if 'scale' in transformations:
        raw_data = pd.read_csv(raw_data_path, parse_dates=True, index_col='Date')
        
        # Initialize a scaler (even though we don't use it for transforming)
        scaler = StandardScaler()
        
        # Fit the scaler on the original raw data to get the original mean and std
        scaler.fit(raw_data[data.columns])
        
        # Reverse the scaling using the mean and scale (std) from the original raw data
        data[data.columns] = scaler.inverse_transform(data)


    # 5. Apply Percentage Change
    if 'pct_change' in transformations:
        for col in data.columns:
            initial_value = data[col].iloc[0]

            reversed_series = (data[col] + 1).cumprod() * initial_value
            reversed_data[col] = reversed_series
            reversed_data.ffill()
        transformations_applied.append('pct_change')

    # 6. Apply ADF-based transformations
    if 'adf' in transformations:
        data = reverse_best_transformations(data)
        transformations_applied.append('adf')

    # 7. Cap outliers
    if 'cap' in transformations:
        skip = 'yes'
        # We can't really do anything
        # data = cap_outliers(data, cap_factor=3.0)
        # transformations_applied.append('cap')

    # 8. Robust Scaler
    if 'robust' in transformations:
        scaler = RobustScaler()
        scaler.fit(raw_data[data.columns])  # Fit on the original data
        data[data.columns] = scaler.inverse_transform(data)


    # 9. Quantile Transformer
    if 'quantile' in transformations_applied:
        scaler = QuantileTransformer(n_quantiles=min(len(raw_data), 1000))
        scaler.fit(raw_data[data.columns])  # Fit on the original data
        data[data.columns] = scaler.inverse_transform(data)

    
    # 10. Power Transformer
    if 'power' in transformations_applied:
        scaler = PowerTransformer()
        scaler.fit(raw_data[data.columns])  # Fit on the original data
        data[data.columns] = scaler.inverse_transform(data)

    
    # 11. Log All
    if 'log_all' in transformations:
        columns_to_transform = data.columns
        data = reverse_log_transformations(data, columns_to_transform)
        transformations_applied.append('log_all')
    
    # 12. Sliding Window Log Transformation
    if 'sliding_window_log' in transformations:
        data = reverse_sliding_window_log(data)
        transformations_applied.append('sliding_window_log')

    # 13. Selective Log Transformation
    if 'selective_log' in transformations:
        threshold = 1.0  # Choose your threshold here
        data = reverse_selective_logging(data, threshold)
        transformations_applied.append('selective_log')

    # 14. Relative Transformations (Normalized Differences)
    if 'relative_transform' in transformations:
        data = reverse_relative_transform(data)
        transformations_applied.append('relative_transform')

    # 15. Local Smoothing Before Logging
    if 'local_smooth' in transformations:
        skip = 'yes'
        # we cannot really reverse local smooth, just leave it alone, skip it, return nothing
        # data = reverse_local_smoothing(data)
        # transformations_applied.append('local_smooth')

    # 16. Soft Clipping for Outliers
    if 'soft_clipping' in transformations:
        threshold = 10  # Set an appropriate threshold
        data = reverse_soft_clipping(data, threshold)
        transformations_applied.append('soft_clipping')
# # Test the data after reversing the transformations
# for transformation in transformation_combinations:
#     # 2. Split the data into features and target
#     targets = ['M1','WTISPLC','CPIAUCSL','HOUST']


 entire list: cap
 individual transformation : impute
 individual transformation : deflate
 individual transformation : log_all
 individual transformation : scale
 individual transformation : pct_change
 individual transformation : adf
 individual transformation : cap


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/testing\\transformed_data__impute__deflate__log_all__scale__pct_change__adf__cap.csv'

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit()
data = pd.read_csv(r'C:\Users\nabounaser\OneDrive - Ejada Systems\EJADA\MacroEconomicAPI\data\processed\testing\transformed_data__impute__adf__robust__sliding_window_log.csv',index_col='Date',parse_dates=True)
split_data = data
print(tscv)
df = pd.DataFrame

targets = ['M1','WTISPLC','CPIAUCSL','HOUST']
# data['index'] = 
for target in targets:
    for tr_index, val_index in tscv.split(data):
        print(tr_index)
        X_train = data[!=target].iloc[tr_index]
        # X_tr, X_val = data[tr_index], data[val_index]
        # y_tr, y_val = y_train[tr_index], 
        # print(tr_index)
        # print(val_index)

In [319]:
targets = ['M1REAL','WTISPLC','CPIAUCSL','HOUST']
features = data.columns
files = os.listdir('data/processed/testing')
xgboost_params = {'max_depth': 9, 'learning_rate': 0.016810144010995204, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5865511380196138, 'colsample_bytree': 0.9177433017782509, 'reg_alpha': 0.012513778476694921, 'reg_lambda': 8.577576991968611e-05}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832469304666387, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402007213047204, 'colsample_bytree': 0.9405245932820381, 'reg_alpha': 0.000756445935013929, 'reg_lambda': 0.0002590470172405959}
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
bst = XGBRegressor(**xgboost_params)
lgbm = LGBMRegressor(**lightgbm_params)
model_type_s = pd.Series(dtype=str)
mse_score_s = pd.Series(dtype=float)
rmse_score_s = pd.Series(dtype=float)
CV_round_s = pd.Series(dtype=int)
target_s = pd.Series(dtype=str)
file_s  = pd.Series(dtype=str)
for file in files:
    data= pd.read_csv('data/processed/testing/'+file,index_col='Date',parse_dates=True)
    # Section removing infs:
    columns_w_inf = []
    print(f"file : {file}")
    for col in data.columns:
        mean = data[col].mean()
        for i in range(len(data)):
            if data.iloc[i][col] > mean*1000000000000:
                if col not in columns_w_inf:
                    columns_w_inf.append(col)
                    # print("Column",col,"has a value too large, on row",i)
    features = list(data.columns)
    new_list = features
    for item in columns_w_inf:
        # print(f"Column: {item} was found with overly large or inf values")
        new_list.remove(item)
    # display(new_list)
    data= data[new_list]
    for item in columns_w_inf:
        # print(f"Column: {item} was found with overly large or inf values")
        if item in new_list:
            new_list.remove(item)
    data = data[new_list]
    # data = data.reset_index()
    for target in targets:
        if target not in new_list:
            # print(f"NEW LIST:",new_list)
            # print(f"target: {target}")
            # print("SKIPPED NOW")
            continue
        for tr_index, val_index in tscv.split(data):
            X_train = data.loc[:,data.columns!= target].iloc[tr_index]
            # display(tr_index)
            # display(val_index)
            X_test = data.loc[:,data.columns!= target].iloc[val_index]
            y_test = data[target].iloc[val_index]
            y_train = data[target].iloc[tr_index]
            bst.fit(X_train,y_train)
            xgb_preds = bst.predict(X_test)
            # print(xgb_preds)
            xgb_mse = mean_squared_error(xgb_preds,y_test)
            xgb_rmse = (xgb_mse)**0.5
            # Append the results to the SERIES
            model_type_s = pd.concat([model_type_s, pd.Series(['XGBoost'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([xgb_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([xgb_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s,pd.Series([file])],ignore_index=True)
            # Appending to CV_round_s based on conditions
            if (tr_index > 40).any() and (tr_index < 75).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([1])], ignore_index=True)
            elif (tr_index > 79).any() and (tr_index < 110).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([2])], ignore_index=True)
            elif (tr_index > 120).any() and (tr_index < 150).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([3])], ignore_index=True)
            elif (tr_index > 150).any() and (tr_index < 190).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([4])], ignore_index=True)
            elif (tr_index > 195).any() and (tr_index < 222).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([5])], ignore_index=True)
            else: 
                CV_round_s = pd.concat([CV_round_s, pd.Series([0])], ignore_index=True)
            lgbm.fit(X_train,y_train)
            lgbm_pred = lgbm.predict(X_test)
            lgbm_mse = mean_squared_error(xgb_preds,y_test)
            lgbm_rmse = (lgbm_mse)**0.5
            # Append the results to the SERIES
            model_type_s = pd.concat([model_type_s, pd.Series(['LightGBM'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([lgbm_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([lgbm_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s,pd.Series([file])],ignore_index=True)
            if (tr_index > 40).any() and (tr_index < 75).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([1])], ignore_index=True)
            elif (tr_index > 79).any() and (tr_index < 110).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([2])], ignore_index=True)
            elif (tr_index > 120).any() and (tr_index < 150).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([3])], ignore_index=True)
            elif (tr_index > 150).any() and (tr_index < 190).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([4])], ignore_index=True)
            elif (tr_index > 195).any() and (tr_index < 222).any():
                CV_round_s = pd.concat([CV_round_s, pd.Series([5])], ignore_index=True)
            else: 
                CV_round_s = pd.concat([CV_round_s, pd.Series([0])], ignore_index=True)
            # X_tr, X_val = data[tr_index], data[val_index]
            # y_tr, y_val = y_train[tr_index], 
            # print(tr_index)
            # print(val_index)

results_df = pd.DataFrame()
results_df['Model Type'] = model_type_s
results_df['MSE Score']  = mse_score_s
results_df['RMSE Score'] = rmse_score_s
results_df['CV round'] = CV_round_s
results_df['File'] = file_s

display(results_df)

# results_df.to_csv("results/final_reversed_data_results.csv")



file : transformed_data__impute.csv
file : transformed_data__impute__adf.csv
file : transformed_data__impute__adf__log_all__local_smooth.csv
file : transformed_data__impute__adf__log_all__soft_clipping.csv
file : transformed_data__impute__adf__power__sliding_window_log.csv
file : transformed_data__impute__adf__quantile__local_smooth.csv
file : transformed_data__impute__adf__quantile__soft_clipping.csv
file : transformed_data__impute__adf__relative_transform__soft_clipping.csv
file : transformed_data__impute__adf__robust__local_smooth.csv
file : transformed_data__impute__adf__robust__sliding_window_log.csv
file : transformed_data__impute__adf__sliding_window_log__soft_clipping.csv
file : transformed_data__impute__adf__soft_clipping.csv
file : transformed_data__impute__deflate__pct_change.csv
file : transformed_data__impute__deflate__pct_change__log_all.csv
file : transformed_data__impute__deflate__scale__pct_change__adf__cap__log_all.csv
file : transformed_data__impute__deflate__scale__

,Model Type,MSE Score,RMSE Score,CV round,File
0,XGBoost,3.593519e+03,5.994597e+01,1,transformed_data__impute.csv
1,LightGBM,3.593519e+03,5.994597e+01,1,transformed_data__impute.csv
2,XGBoost,4.625687e+04,2.150741e+02,1,transformed_data__impute.csv
3,LightGBM,4.625687e+04,2.150741e+02,1,transformed_data__impute.csv
4,XGBoost,4.555491e+04,2.134360e+02,1,transformed_data__impute.csv
...,...,...,...,...,...
1825,LightGBM,1.662958e+12,1.289557e+06,1,transformed_data__impute__scale__robust__quant...
1826,XGBoost,4.633268e+11,6.806811e+05,1,transformed_data__impute__scale__robust__quant...
1827,LightGBM,4.633268e+11,6.806811e+05,1,transformed_data__impute__scale__robust__quant...
1828,XGBoost,1.210364e+10,1.100166e+05,1,transformed_data__impute__scale__robust__quant...


In [320]:
# Filtering results_df based on the file name
# results_df.to_csv('results/process_data_eval_without_reversal.csv')


In [134]:
from xgboost import XGBRegressor

bst = XGBRegressor(**xgboost_params)
lightgbm = lgbm()

# CREATIN REVERSAL FUNCTION:

In [65]:
import pandas as pd
import numpy as np

def reverse_deflate_nominal_values(df, raw_data, cpi_col_name, columns_to_deflate, target):
    """
    Inflates the deflated values in the specified columns of the series using the original CPI column from raw data.

    :param df: Series containing the values to inflate
    :param cpi_col_name: Name of the CPI column
    :param columns_to_deflate: List of column names to deflate
    :return: Series with inflated values
    """
    original_cpi = raw_data[cpi_col_name]
    if target in columns_to_deflate:
        df = df * original_cpi / 100
    return df

def reverse_log_transformations(df, columns_to_transform):
    """
    Reverses log1p transformations applied to the specified columns with overflow protection.

    :param df: DataFrame to reverse the log transformation on.
    :param columns_to_transform: List of columns to reverse.
    :return: DataFrame with log transformation reversed.
    """
    # Define a maximum value to prevent overflow
    MAX_EXP_VALUE = 700  # This prevents np.exp from causing overflow

    for col in columns_to_transform:
        # Clip the values before applying expm1 to prevent overflow
        df[col] = np.where(df[col] > 0, np.expm1(np.clip(df[col] / 100, a_min=None, a_max=MAX_EXP_VALUE)), df[col])
        # Handle NaNs by forward-filling and backward-filling (optional, based on your workflow)
        df[col].fillna(method='ffill', inplace=True)
        df[col].fillna(method='bfill', inplace=True)

    return df


def reverse_all_log_transformations(df):
    """
    Reverses log transformations for the entire series with overflow protection.

    :param df: Series to reverse log transformation
    :return: Series with log transformation reversed
    """
    # Define a maximum value to prevent overflow
    MAX_EXP_VALUE = 700  # Prevents np.exp from causing overflow

    # Apply the reverse log transformation with clipping to prevent overflow
    df = pd.Series(np.where(df > 0, np.expm1(np.clip(df / 100, a_min=None, a_max=MAX_EXP_VALUE)), df), index=df.index)
    
    return df


def reverse_best_transformations(df, raw_data, target):
    """
    Reverses the best transformation applied to the data based on a saved file.

    :param df: Series containing the transformed data
    :param raw_data: The original raw data
    :param target: Target column to reverse
    :return: Series with best transformation reversed
    """
    transformation_results = pd.read_csv('best_transformations.csv')
    best_method_row = transformation_results[transformation_results['Target'] == target]

    if best_method_row.empty:
        raise ValueError(f"No transformation method found for target {target}")

    best_method = best_method_row['Best Method'].values[0]

    if best_method == 'Simple Differencing':
        first_value = raw_data[target].iloc[0]
        df = df.cumsum() + first_value
    elif best_method == 'Rolling Mean Subtraction':
        rolling_mean = raw_data[target].rolling(window=7).mean()
        df = (df + rolling_mean).ffill()
    elif best_method == 'Rolling Mean Subtraction + Differencing':
        rolling_mean = raw_data[target].rolling(window=7).mean()
        first_value = raw_data[target].iloc[0]
        reversed_diff = df.cumsum() + first_value
        df = (reversed_diff + rolling_mean).ffill()
    return df

def reverse_sliding_window_log(df, window_size=12):
    """
    Reverses the sliding window log transformation.

    :param df: Series containing the transformed data
    :param window_size: The window size used during the transformation
    :return: Series with sliding window log transformation reversed
    """
    reversed_data = df.copy()
    for i in range(window_size - 1, len(df)):
        transformed_window = df.iloc[i - window_size + 1: i + 1]
        reversed_window = np.expm1(transformed_window)
        reversed_data.iloc[i - window_size + 1: i + 1] = reversed_window

    reversed_data.fillna(method='ffill', inplace=True)
    reversed_data.fillna(method='bfill', inplace=True)
    return reversed_data

def reverse_selective_logging(df, threshold):
    """
    Reverses selective logging transformation.

    :param df: Series containing the transformed data
    :param threshold: Threshold used during the transformation
    :return: Series with selective logging reversed
    """
    df = pd.Series(np.where(df > threshold, np.expm1(df), df), index=df.index)
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)
    return df

def reverse_relative_transform(df, target):
    """
    Reverses relative transform applied to the data.

    :param df: Series containing the transformed data
    :param target: Target column to reverse
    :return: Series with relative transform reversed
    """
    raw_data = pd.read_csv('data/raw/raw_data.csv', index_col='Date', parse_dates=True)
    initial_value = raw_data[target].iloc[0]
    df = df.cumsum() + initial_value
    df = df * (raw_data[target].shift(1) + 1e-9)
    return df

def reverse_rolling_mean(df, window_size=5):
    """
    Reverses the rolling mean applied to the data.

    :param df: Series containing the transformed data
    :param window_size: Window size used during the rolling mean
    :return: Series with rolling mean reversed
    """
    reversed_data = pd.Series(index=df.index, dtype=float)

    for i in range(len(df)):
        start_idx = max(0, i - window_size + 1)
        end_idx = i + 1

        if end_idx > start_idx:
            window_mean = df.iloc[start_idx:end_idx].mean()
            if i > 0:
                previous_value = reversed_data.iloc[i - 1] if not pd.isna(reversed_data.iloc[i - 1]) else df.iloc[i]
            else:
                previous_value = df.iloc[i]
            reversed_data.iloc[i] = df.iloc[i] * window_size - (window_mean * (window_size - 1))

    return reversed_data

def reverse_soft_clipping(df, threshold, target, n=1):
    """
    Reverses soft clipping applied to the data.

    :param df: Series containing the transformed data
    :param threshold: Threshold used during the soft clipping
    :param target: Target column to reverse
    :param n: Exponent used during the soft clipping
    :return: Series with soft clipping reversed
    """
    raw_data = pd.read_csv('data/raw/raw_data.csv', index_col='Date', parse_dates=True)
    df = df * (1 + (raw_data[target] / threshold) ** n)
    return df

import numpy as np


def safe_expm(x):
    """
    Safely compute the exponential to avoid overflow.
    Clipping the input to a maximum threshold to prevent np.exp from producing infinity.
    """
    # Maximum value for exp to avoid overflow
    MAX_EXP_VALUE = 700
    x_clipped = np.clip(x, a_min=None, a_max=MAX_EXP_VALUE)  # Clip to prevent overflow
    return np.exp(x_clipped)


def safe_cumprod(x):
    """
    Safely compute cumulative product to avoid overflow.
    Clipping the input values to prevent extremely large cumulative products.
    """
    # Maximum value for cumprod to avoid overflow
    MAX_CUMPROD_VALUE = 1e8  # Reduce the threshold to a lower, safer value
    MIN_CUMPROD_VALUE = 1e-8  # Set a reasonable lower bound to prevent underflow

    # Clip the values to prevent overflow or underflow during cumulative product
    x_clipped = np.clip(x, a_min=MIN_CUMPROD_VALUE, a_max=MAX_CUMPROD_VALUE)
    
    # Compute the cumulative product on the clipped values
    return np.cumprod(x_clipped)





In [66]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
# from funcs.process_data_funcs import (
#     reverse_deflate_nominal_values,
#     reverse_log_transformations,
#     reverse_best_transformations,
#     reverse_all_log_transformations,
#     reverse_sliding_window_log,
#     reverse_selective_logging,
#     reverse_relative_transform,
#     reverse_soft_clipping
# )

def reverse_transformed_data(y_pred, y_test, transformations):
    """
    Args:
        y_pred: The predictions of our model, to be reversed back to
                the form of original data, for the mse calculation to be performed.
        y_test: The actual transformed data.
        transformations: A list containing strings, listing all transformations
                         applied to a certain dataset/file.
    """
    raw_data = pd.read_csv('data/raw/raw_data.csv',index_col='Date',parse_dates=True)

    # 16. Soft Clipping for Outliers
    if 'soft_clipping' in transformations:
        threshold = 10  # Set an appropriate threshold
        y_test = reverse_soft_clipping(y_test, threshold,target)
        y_pred = reverse_soft_clipping(y_pred, threshold,target)

    # 15. Local Smoothing Before Logging
    if 'local_smooth' in transformations:
        skip = 'yes'
        # We cannot really reverse local smoothing, so skip it.

    # 14. Relative Transformations (Normalized Differences)
    if 'relative_transform' in transformations:
        y_test = reverse_relative_transform(y_test, target)
        y_pred = reverse_relative_transform(y_pred, target)

    # 13. Selective Log Transformation
    if 'selective_log' in transformations:
        threshold = 1.0  # Choose your threshold here
        y_test = reverse_selective_logging(y_test, threshold)
        y_pred = reverse_selective_logging(y_pred, threshold)

    # 12. Sliding Window Log Transformation
    if 'sliding_window_log' in transformations:
        y_test = reverse_sliding_window_log(y_test)
        y_pred = reverse_sliding_window_log(y_pred)

    # 11. Log All
    if 'log_all' in transformations:
        y_test = reverse_all_log_transformations(y_test)
        y_pred = reverse_all_log_transformations(y_pred)

    # 10. Power Transformer
    if 'power' in transformations:
        scaler = PowerTransformer()
        scaler.fit(raw_data[target].values.reshape(-1, 1))  # Fit on the original data
        y_test_transformed = scaler.inverse_transform(y_test.values.reshape(-1, 1)).flatten()  # Flatten the array
        y_pred_transformed = scaler.inverse_transform(y_pred.values.reshape(-1, 1)).flatten()  # Flatten the array

        y_test = pd.Series(y_test_transformed, index=y_test.index)
        y_pred = pd.Series(y_pred_transformed, index=y_pred.index)

    # 9. Quantile Transformer
    if 'quantile' in transformations:
        scaler = QuantileTransformer(n_quantiles=min(len(raw_data), 1000))
        scaler.fit(raw_data[target].values.reshape(-1, 1))  # Fit on the original data
        y_test_transformed = scaler.inverse_transform(y_test.values.reshape(-1, 1)).flatten()  # Flatten the array
        y_pred_transformed = scaler.inverse_transform(y_pred.values.reshape(-1, 1)).flatten()  # Flatten the array

        y_test = pd.Series(y_test_transformed, index=y_test.index)
        y_pred = pd.Series(y_pred_transformed, index=y_pred.index)


    # 8. Robust Scaler
    if 'robust' in transformations:
        scaler = RobustScaler()
        scaler.fit(raw_data[target].values.reshape(-1,1))  # Fit on the original data
        y_test_transformed = scaler.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        y_pred_transformed = scaler.inverse_transform(y_pred.values.reshape(-1,1)).flatten()

        y_test = pd.Series(y_test_transformed,index=y_test.index)
        y_pred = pd.Series(y_pred_transformed,index=y_pred.index)



    # 7. Cap outliers
    if 'cap' in transformations:
        skip = 'yes'
        # We can't really do anything here.

    # 6. Apply ADF-based transformations
    if 'adf' in transformations:
        y_pred = reverse_best_transformations(y_pred, raw_data, target)
        y_test = reverse_best_transformations(y_test, raw_data, target)

    if 'pct_change' in transformations:
        initial_value = raw_data[target].iloc[0]
        
        # Apply safe_cumprod with overflow handling
        y_pred_reversed_series = safe_cumprod(y_pred + 1) * initial_value
        y_test_reversed_series = safe_cumprod(y_test + 1) * initial_value
        
        # Fill missing values (forward-fill and backward-fill)
        y_pred = y_pred_reversed_series.ffill()
        y_test = y_test_reversed_series.ffill()

    # 4. Standardize/Normalize the Data
    if 'scale' in transformations:
        raw_data = pd.read_csv("data/raw/raw_data.csv", parse_dates=True, index_col='Date')
        scaler = StandardScaler()
        scaler.fit(raw_data[target].values.reshape(-1,1))  # Fit on the original data
        y_test_transformed = scaler.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        y_pred_transformed = scaler.inverse_transform(y_pred.values.reshape(-1,1)).flatten()

        y_test = pd.Series(y_test_transformed,index=y_test.index)
        y_pred = pd.Series(y_pred_transformed,index=y_pred.index)
  
    # 3. Apply logarithmic transformations
    if 'log' in transformations and 'log_all' not in transformations:
        columns_to_transform = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS',
            'GCE', 'FGCE', 'HOUST', 'DSPI', 'M1', 'M1V', 'M2',
            'WTISPLC'
        ]
        y_pred = reverse_log_transformations(y_pred, columns_to_transform, target)
        y_test = reverse_log_transformations(y_test, columns_to_transform, target)

    # 2. Deflate nominal values
    if 'deflate' in transformations:
        cpi_col_name = 'CPIAUCSL'
        columns_to_deflate = [
            'GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE',
            'FGCE', 'DSPI'
        ]
        y_pred = reverse_deflate_nominal_values(y_pred, raw_data, cpi_col_name, columns_to_deflate, target)
        y_test = reverse_deflate_nominal_values(y_test, raw_data, cpi_col_name, columns_to_deflate, target)

    # 1. Impute missing values
    if 'impute' in transformations:
        skip = 'yes'
        # Since we cannot have NaN's, we will never reverse the imputation of data.

    y_pred = pd.Series(CubicSpline(y_pred.dropna().index, y_pred.dropna())(y_pred.index), index=y_pred.index)
    y_test = pd.Series(CubicSpline(y_test.dropna().index, y_test.dropna())(y_test.index), index=y_test.index)

    return y_pred, y_test


In [63]:
# v.1.0.0: traditional inf handling, mean * 10000000000
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from funcs.process_data_funcs import (
    impute_missing_values_spline, deflate_nominal_values, apply_log_transformations,
    apply_best_transformations, cap_outliers
)
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from sklearn.neighbors import KNeighborsRegressor
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data
from funcs.dvc_funcs import dagshub_initialization
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import QuantileTransformer, PowerTransformer


targets = ['M1REAL','WTISPLC','CPIAUCSL','HOUST']
files = os.listdir('data/processed/testing')
xgboost_params = {'max_depth': 9, 'learning_rate': 0.016810144010995204, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5865511380196138, 'colsample_bytree': 0.9177433017782509, 'reg_alpha': 0.012513778476694921, 'reg_lambda': 8.577576991968611e-05}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832469304666387, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402007213047204, 'colsample_bytree': 0.9405245932820381, 'reg_alpha': 0.000756445935013929, 'reg_lambda': 0.0002590470172405959}
bst = XGBRegressor(**xgboost_params,verbosity = 0)
lgbm = LGBMRegressor(**lightgbm_params,verbose=-1)
model_type_s = pd.Series(dtype=str)
mse_score_s = pd.Series(dtype=float)
rmse_score_s = pd.Series(dtype=float)
CV_round_s = pd.Series(dtype=int)
target_s = pd.Series(dtype=str)
file_s  = pd.Series(dtype=str)
for file in files:
    data= pd.read_csv('data/processed/testing/'+file,index_col='Date',parse_dates=True)
    features = data.columns
    # Section removing infs and very large values
    columns_w_inf = []
    print(f"file : {file}")
    for col in data.columns:
        mean = data[col].mean()
        for i in range(len(data)):
            if data.iloc[i][col] > mean*1000000000000:
                if col not in columns_w_inf:
                    columns_w_inf.append(col)
                    # print("Column",col,"has a value too large, on row",i)
    features = list(data.columns)
    new_list = features
    for item in columns_w_inf:
        # print(f"Column: {item} was found with overly large or inf values")
        new_list.remove(item)
    data= data[new_list]
    for item in columns_w_inf:
        # print(f"Column: {item} was found with overly large or inf values")
        if item in new_list:
            new_list.remove(item)
    data = data[new_list]

    # Creating a list containing all transformations applied on the file
    transformations = []
    transformation_strings = ['impute', 'deflate', 'log', 'scale', 'pct_change', 'adf', 'cap', 'robust', 'quantile', 'power', 'log_all', 'sliding_window_log', 'selective_log', 'soft_clipping', 'local_smooth', 'relative_transform']

    file_name_parts = file.split('__')  # Split the filename at '__'
    for part in file_name_parts:
        if part in transformation_strings:
            transformations.append(part)

    print(f" entire list: {transformations} in file {file}")
    for target in targets:
        tscv = TimeSeriesSplit()
        if target not in new_list:
            # print(f"NEW LIST:",new_list)
            # print(f"target: {target}")
            # print("SKIPPED NOW")
            continue
        for round_number, (tr_index, val_index) in enumerate(tscv.split(data),start=1):
            X_train = data.loc[:,data.columns!= target].iloc[tr_index]
            # display(tr_index)
            # display(val_index)
            X_test = data.loc[:,data.columns!= target].iloc[val_index]
            y_test = data[target].iloc[val_index]
            y_train = data[target].iloc[tr_index]

            # XGBoost Section
            bst.fit(X_train,y_train)
            xgb_preds = bst.predict(X_test)
            xgb_preds = pd.Series(xgb_preds,index=y_test.index)
            # Before evaluation, we reverse transformations
            # Display the type of xgb_y_pred and y_test
            xgb_y_pred, xgb_y_test = reverse_transformed_data(y_pred=xgb_preds, y_test=y_test.copy(), transformations=transformations)
            xgb_mse = mean_squared_error(xgb_preds,y_test)
            xgb_rmse = (xgb_mse)**0.5
            # Append the results to the SERIES
            model_type_s = pd.concat([model_type_s, pd.Series(['XGBoost'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([xgb_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([xgb_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s,pd.Series([file])],ignore_index=True)
            CV_round_s = pd.concat([CV_round_s,pd.Series([round_number])],ignore_index=True)

            # LightGBM Section
            lgbm.fit(X_train,y_train)
            lgbm_pred = lgbm.predict(X_test)
            lgbm_pred = pd.Series(lgbm_pred,index=y_test.index)

            lgbm_y_pred, lgbm_y_test = reverse_transformed_data(y_pred=lgbm_pred, y_test=y_test.copy(), transformations=transformations)
            lgbm_mse = mean_squared_error(lgbm_pred,y_test)
            lgbm_rmse = (lgbm_mse)**0.5   
            # Append the results to the SERIES
            model_type_s = pd.concat([model_type_s, pd.Series(['LightGBM'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([lgbm_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([lgbm_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s,pd.Series([file])],ignore_index=True)
            CV_round_s = pd.concat([CV_round_s,pd.Series([round_number])],ignore_index=True)


            # X_tr, X_val = data[tr_index], data[val_index]
            # y_tr, y_val = y_train[tr_index], 
            # print(tr_index)
            # print(val_index)

results_df = pd.DataFrame()
results_df['Model Type'] = model_type_s
results_df['MSE Score']  = mse_score_s
results_df['RMSE Score'] = rmse_score_s
results_df['CV round'] = CV_round_s
results_df['File'] = file_s

display(results_df)

results_df.to_csv("results/final_reversed_data_results.csv")

file : transformed_data__impute.csv


 entire list: [] in file transformed_data__impute.csv
file : transformed_data__impute__adf.csv
 entire list: ['impute'] in file transformed_data__impute__adf.csv
file : transformed_data__impute__adf__log_all__local_smooth.csv
 entire list: ['impute', 'adf', 'log_all'] in file transformed_data__impute__adf__log_all__local_smooth.csv
file : transformed_data__impute__adf__log_all__soft_clipping.csv
 entire list: ['impute', 'adf', 'log_all'] in file transformed_data__impute__adf__log_all__soft_clipping.csv
file : transformed_data__impute__adf__power__sliding_window_log.csv
 entire list: ['impute', 'adf', 'power'] in file transformed_data__impute__adf__power__sliding_window_log.csv
file : transformed_data__impute__adf__quantile__local_smooth.csv
 entire list: ['impute', 'adf', 'quantile'] in file transformed_data__impute__adf__quantile__local_smooth.csv
file : transformed_data__impute__adf__quantile__soft_clipping.csv
 entire list: ['impute', 'adf', 'quantile'] in file transformed_data__imp

c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\numpy\core\fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)


ValueError: `y` must contain only finite values.

In [69]:
# v.2.0.0:
# #  Enhanced inf/nan handling using np.isinf and np.isnan
# # ALSO will be able to handle infinities after the fact
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import os
import pandas as pd
from scipy.interpolate import CubicSpline
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from funcs.process_data_funcs import (
    impute_missing_values_spline, deflate_nominal_values, apply_log_transformations,
    apply_best_transformations, cap_outliers
)
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from sklearn.neighbors import KNeighborsRegressor
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data
from funcs.dvc_funcs import dagshub_initialization
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import QuantileTransformer, PowerTransformer

# Targets and files
targets = ['M1REAL','WTISPLC','CPIAUCSL','HOUST']
files = os.listdir('data/processed/testing')

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.016810144010995204, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5865511380196138, 'colsample_bytree': 0.9177433017782509, 'reg_alpha': 0.012513778476694921, 'reg_lambda': 8.577576991968611e-05}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832469304666387, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402007213047204, 'colsample_bytree': 0.9405245932820381, 'reg_alpha': 0.000756445935013929, 'reg_lambda': 0.0002590470172405959}

# Initialize models
bst = XGBRegressor(**xgboost_params, verbosity=0)
lgbm = LGBMRegressor(**lightgbm_params, verbose=-1)

# Initialize dataframes for storing results
model_type_s = pd.Series(dtype=str)
mse_score_s = pd.Series(dtype=float)
mse_raw_score_s = pd.Series(dtype=float)
data_corr_score_s = pd.Series(dtype=float)
rmse_score_s = pd.Series(dtype=float)
CV_round_s = pd.Series(dtype=int)
target_s = pd.Series(dtype=str)
file_s = pd.Series(dtype=str)


# Helper function to remove infinities and NaNs
def remove_infs_nans(data):
    data = data.replace([np.inf, -np.inf], np.nan)  # Replace infinities with NaN
    data = data.dropna()  # Remove rows with NaN values
    return data
raw = pd.read_csv('data/raw/raw_data.csv',index_col='Date',parse_dates=True)
# Iterate through files
for file in files:
    data = pd.read_csv(f'data/processed/testing/{file}', index_col='Date', parse_dates=True)
    features = data.columns
    
    # Handle infinities and NaNs before model training
    data = remove_infs_nans(data)
    
    print(f"file: {file}")

    # Creating a list containing all transformations applied to the file
    transformations = []
    transformation_strings = ['impute', 'deflate', 'log', 'scale', 'pct_change', 'adf', 'cap', 'robust', 'quantile', 'power', 'log_all', 'sliding_window_log', 'selective_log', 'soft_clipping', 'local_smooth', 'relative_transform']
    # remove the .csv from the strings

    file_name_parts = file.split('__')
    print(f"file name parts: {file_name_parts}")

    # Split the filename by '__'
    file_name_parts = file.split('__')

    # Check each part of the filename
    for i, part in enumerate(file_name_parts):
            # If the part is the last one (which may have .csv), strip the .csv
        if i == len(file_name_parts) - 1 and part.endswith('.csv'):
            part = part.replace('.csv', '')

        if part in transformation_strings:
            transformations.append(part)

    print(f"Entire list: {transformations} in file {file}")

    for target in targets:
        tscv = TimeSeriesSplit()
        if target not in data.columns:
            continue
        raw_data = raw[target]

        for round_number, (tr_index, val_index) in enumerate(tscv.split(data), start=1):
            X_train = data.loc[:, data.columns != target].iloc[tr_index]
            X_test = data.loc[:, data.columns != target].iloc[val_index]
            y_test_real = data[target].iloc[val_index]
            y_train = data[target].iloc[tr_index]
            raw_test = raw_data.iloc[val_index]

            # XGBoost Section
            y_test = y_test_real.copy()
            bst.fit(X_train, y_train)
            xgb_preds = bst.predict(X_test)
            xgb_preds = pd.Series(xgb_preds, index=y_test.index)
            
            # Reverse transformations
            try:
                xgb_y_pred, xgb_y_test = reverse_transformed_data(y_pred=xgb_preds, y_test=y_test.copy(), transformations=transformations)
            except ValueError:
                print(f"ValueError encountered in file: {file},performing another round of removing infinities")
                xgb_preds = remove_infs_nans(xgb_preds)
                y_test = remove_infs_nans(y_test)
                continue

            try: 
                xgb_y_pred = pd.Series(CubicSpline(xgb_y_pred.dropna().index, xgb_y_pred.dropna())(xgb_y_pred.index), index=xgb_y_pred.index)
            except ValueError:
                xgb_y_pred = remove_infs_nans(xgb_y_pred)
                continue

            try:
                xgb_y_test = pd.Series(CubicSpline(xgb_y_test.dropna().index, xgb_y_test.dropna())(xgb_y_test.index), index=xgb_y_test.index)
            except ValueError:
                xgb_y_test = remove_infs_nans(xgb_y_test)
                continue
            xgb_mse = mean_squared_error(xgb_y_pred, xgb_y_test)

            # Immediately reindex raw_test to align with xgb_y_pred
            aligned_raw_test = raw_test.reindex(xgb_y_pred.index)

            # Calculate MSE after reindexing raw_test
            aligned_raw_test = pd.Series(CubicSpline(aligned_raw_test.dropna().index, aligned_raw_test.dropna())(aligned_raw_test.index), index=aligned_raw_test.index)
            xgb_raw_mse = mean_squared_error(xgb_y_pred, aligned_raw_test)

            xgb_rmse = np.sqrt(xgb_mse)
            data_corr = np.corrcoef(xgb_y_test, aligned_raw_test)[0, 1]

            # Append results
            model_type_s = pd.concat([model_type_s, pd.Series(['XGBoost'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([xgb_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([xgb_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s, pd.Series([file])], ignore_index=True)
            CV_round_s = pd.concat([CV_round_s, pd.Series([round_number])], ignore_index=True)
            mse_raw_score_s = pd.concat([mse_raw_score_s, pd.Series([xgb_raw_mse])], ignore_index=True)
            data_corr_score_s = pd.concat([data_corr_score_s, pd.Series([data_corr])], ignore_index=True)

            # LightGBM Section
            y_test = y_test_real.copy()
            lgbm.fit(X_train, y_train)
            lgbm_pred = lgbm.predict(X_test)
            lgbm_pred = pd.Series(lgbm_pred, index=y_test.index)

            # Reverse transformations
            try:
                lgbm_y_pred, lgbm_y_test = reverse_transformed_data(y_pred=lgbm_pred, y_test=y_test.copy(), transformations=transformations)
            except ValueError:
                print(f"ValueError encountered in file: {file},performing another round of removing infinities")
                lgbm_y_pred = remove_infs_nans(lgbm_y_pred)
                y_test = remove_infs_nans(y_test)
                continue
            
            try:
                lgbm_y_pred = pd.Series(CubicSpline(lgbm_y_pred.dropna().index, lgbm_y_pred.dropna())(lgbm_y_pred.index), index=lgbm_y_pred.index)
            except ValueError:
                lgbm_y_pred = remove_infs_nans(lgbm_y_pred)
                continue

            try:
                lgbm_y_test = pd.Series(CubicSpline(lgbm_y_test.dropna().index, lgbm_y_test.dropna())(lgbm_y_test.index), index=lgbm_y_test.index)
            except ValueError:
                lgbm_y_test = remove_infs_nans(lgbm_y_test)
                continue

            lgbm_mse = mean_squared_error(lgbm_y_pred, lgbm_y_test)
            aligned_raw_test = raw_test.reindex(lgbm_y_pred.index)
            aligned_raw_test = pd.Series(CubicSpline(aligned_raw_test.dropna().index, aligned_raw_test.dropna())(aligned_raw_test.index), index=aligned_raw_test.index)
            lgbm_raw_mse = mean_squared_error(lgbm_y_pred, aligned_raw_test)
            lgbm_rmse = np.sqrt(lgbm_mse)

            # Append results
            model_type_s = pd.concat([model_type_s, pd.Series(['LightGBM'])], ignore_index=True)
            mse_score_s = pd.concat([mse_score_s, pd.Series([lgbm_mse])], ignore_index=True)
            rmse_score_s = pd.concat([rmse_score_s, pd.Series([lgbm_rmse])], ignore_index=True)
            target_s = pd.concat([target_s, pd.Series([target])], ignore_index=True)
            file_s = pd.concat([file_s, pd.Series([file])], ignore_index=True)
            CV_round_s = pd.concat([CV_round_s, pd.Series([round_number])], ignore_index=True)
            mse_raw_score_s = pd.concat([mse_raw_score_s, pd.Series([lgbm_raw_mse])], ignore_index=True)
            data_corr_score_s = pd.concat([data_corr_score_s, pd.Series([data_corr])], ignore_index=True)
            
# Final results dataframe
results_df = pd.DataFrame({
    'Model Type': model_type_s,
    'MSE Score': mse_score_s,
    'MSE Raw Score': mse_raw_score_s,
    'RMSE Score': rmse_score_s,
    'Data Correlation': data_corr_score_s,
    'CV round': CV_round_s,
    'File': file_s,
})

display(results_df)
results_df.to_csv("results/final_reversed_data_results.csv")


file: transformed_data__impute.csv
file name parts: ['transformed_data', 'impute.csv']
Entire list: ['impute'] in file transformed_data__impute.csv
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
file: transformed_data__impute__adf.csv
file name pa

c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\scipy\interpolate\_cubic.py:780: RuntimeWarning: overflow encountered in scalar multiply
  b[-1] = ((dxr[-1]**2*slope[-2] +
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\scipy\interpolate\_cubic.py:781: RuntimeWarning: overflow encountered in scalar multiply
  (2*d + dxr[-1])*dxr[-2]*slope[-1]) / d)


XGB y_pred NaN: 0 and XGB y_test NaN: 0


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\scipy\interpolate\_cubic.py:765: RuntimeWarning: overflow encountered in scalar multiply
  b[0] = ((dxr[0] + 2*d) * dxr[1] * slope[0] +
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\scipy\interpolate\_cubic.py:766: RuntimeWarning: overflow encountered in scalar multiply
  dxr[0]**2 * slope[1]) / d


XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
file: transformed_data__impute__adf__power__sliding_window_log.csv
file name parts: ['transformed_data', 'impute', 'adf', 'power', 'sliding_window_log.csv']
Entire list: ['impute', 'adf', 'power', 'sliding_window_log'] in file transformed_data__impute__adf__power__sliding_window_log.csv
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test

...

c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\numpy\core\fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)


ValueError encountered in file: transformed_data__impute__deflate__scale__pct_change__adf__log_all.csv,performing another round of removing infinities
XGB y_pred NaN: 0 and XGB y_test NaN: 0


c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\metrics\_regression.py:478: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\metrics\_regression.py:478: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\numpy\lib\function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env\lib\site-packages\sklearn\metrics\_regression.py:478: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)
c:\Users\nabounaser\AppData\Local\miniconda3\envs\autogluon_env

XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
file: transformed_data__impute__deflate__scale__pct_change__log_all.csv
file name parts: ['transformed_data', 'impute', 'deflate', 'scale', 'pct_change', 'log_all.csv']
Entire list: ['impute', 'deflate', 'scale', 'pct_change', 'log_all'] in file transformed_data__impute__deflate__scale__pct_change__log_all.csv
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pred NaN: 0 and XGB y_test NaN: 0
XGB y_pr

KeyboardInterrupt: 